In [12]:
import os
import numpy as np
from tqdm import tqdm
import pandas as pd
import pandapower as pp
from pandapower.powerflow import LoadflowNotConverged
import networkx as nx
import pandapower.networks as pn
import pandapower.topology as top
import pandapower.plotting as plot
from pandapower.control import ConstControl
from pandapower.timeseries import DFData, OutputWriter, run_timeseries
import matplotlib.pyplot as plt
from scipy.stats import norm
from dowhy import CausalModel
from pandapower.pypower.makeYbus import makeYbus
import scipy.linalg
import scipy.sparse as sp

In [13]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

### The Function to check N-1 contingency criterion with the RTS24 Test Network

In [14]:
# -*- coding: utf-8 -*-
"""
RTS-24 with renewables + N-1 reinforcement (root-cause aware)
- Adds renewables (optional) with your Pmax (from real data)
- Per-element cost rows (avoid pandas vectorized assignment error)
- N-1 screening (line/trafo/gen outages) via AC-OPF
- Reinforcement policy:
    * If outage causes OPF infeasible:
        - line outage -> add a parallel line to THAT line
        - trafo outage -> add a parallel transformer to THAT transformer
        - gen outage -> increase max_p_mw for OTHER generators slightly
    * If outage is feasible but has line overloads -> add parallel lines to overloaded lines
- Iterate until N-1 passes or max iterations reached
- Saves final network as JSON

Tested with: Python 3.11, pandapower 2.13+
"""

import copy
import numpy as np
import pandapower as pp
import pandapower.networks as pn

# -----------------------
# (Optional) renewables from your real data
# Fill bus id and Pmax (MW). If not needed, leave list empty.
RENEWABLE_PLANTS = [
    # (bus_id_in_rts24, Pmax_MW)
    # (3, 80.0),
    # (7, 60.0),
    # (16, 50.0),
]

# -----------------------
# Global settings
# -----------------------
VMIN_PU = 0.95
VMAX_PU = 1.05
LINE_LOADING_LIMIT = 100.0    # %
MAX_ITER = 16                 # allow more passes
OPF_TOL = 1e-6

ADD_PARALLEL_LINES = True     # True: add parallel circuits; False: uprate ampacity
LINE_UPRATE_FACTOR = 1.25     # used when not adding parallels
GEN_MAXP_UPFACTOR  = 1.05     # every time we relax gens

# Slack pricier than gens -> prefer local gens first (more realistic)
UNLIMITED_SLACK = False

# (Optional) caps to avoid infinite growth
MAX_PARALLELS_PER_ELEMENT = 3


# -----------------------
# Utilities
# -----------------------
def add_cost_row_per_element(net, element_indices, et, cp0=0.0, cp1=5.0, cp2=0.02):
    """Add poly cost rows one-by-one (avoid vectorized create_poly_cost)."""
    for idx in element_indices:
        pp.create_poly_cost(
            net, element=int(idx), et=et,
            cp0_eur=cp0, cp1_eur_per_mw=cp1, cp2_eur_per_mw2=cp2
        )


def build_base_rts24():
    net = pn.case24_ieee_rts()

    # Voltage bounds
    net.bus["min_vm_pu"] = VMIN_PU
    net.bus["max_vm_pu"] = VMAX_PU

    # Ensure controllable flags
    for df in (net.gen, net.ext_grid):
        if "controllable" not in df.columns:
            df["controllable"] = True
        else:
            df["controllable"] = True

    # Wide bounds for ext_grid (feasibility guard); cost decides dispatch preference
    for col, val in [("min_p_mw", -1e4), ("max_p_mw", 1e4),
                     ("min_q_mvar", -1e4), ("max_q_mvar", 1e4)]:
        if col not in net.ext_grid.columns:
            net.ext_grid[col] = val
        else:
            net.ext_grid[col] = val

    # Ensure generator bounds exist
    if "min_p_mw" not in net.gen.columns:
        net.gen["min_p_mw"] = 0.0
    if "max_p_mw" not in net.gen.columns:
        net.gen["max_p_mw"] = np.maximum(net.gen["p_mw"].values * 1.5, 20.0)
    if "min_q_mvar" not in net.gen.columns:
        net.gen["min_q_mvar"] = -1e3
    if "max_q_mvar" not in net.gen.columns:
        net.gen["max_q_mvar"] = 1e3

    # Ensure line ampacity exists
    if "max_i_ka" not in net.line.columns:
        net.line["max_i_ka"] = 1.0

    # Track how many parallels we added (custom columns)
    if "parallels_added" not in net.line.columns:
        net.line["parallels_added"] = 0
    if "parallels_added" not in net.trafo.columns:
        net.trafo["parallels_added"] = 0

    # Reset poly_cost, then add per-element rows
    if len(net.poly_cost):
        net.poly_cost.drop(net.poly_cost.index, inplace=True)

    # Conventional gens: normal cost
    if len(net.gen):
        add_cost_row_per_element(net, net.gen.index.tolist(), et="gen", cp0=0.0, cp1=5.0, cp2=0.02)

    # External grid: pricier than gens (prefer local gen first)
    if len(net.ext_grid):
        add_cost_row_per_element(net, net.ext_grid.index.tolist(), et="ext_grid",
                                 cp0=0.0,
                                 cp1=(1e-6 if UNLIMITED_SLACK else 8.0),
                                 cp2=(0.0   if UNLIMITED_SLACK else 0.02))
    return net


def add_renewables(net, plants):
    """Add renewable generators at buses with given Pmax; very low marginal cost."""
    new_gen_indices = []
    for bus, pmax in plants:
        gidx = pp.create_gen(
            net, bus=int(bus), p_mw=0.0, vm_pu=1.0,
            min_p_mw=0.0, max_p_mw=float(pmax), name=f"RES@{bus}"
        )
        new_gen_indices.append(gidx)
    if new_gen_indices:
        # cheap renewable cost
        add_cost_row_per_element(net, new_gen_indices, et="gen", cp0=0.0, cp1=0.05, cp2=0.0)


def run_ac_opf(net):
    try:
        pp.runopp(net, calculate_voltage_angles=True, verbose=False, enforce_q_lims=True, delta=OPF_TOL)
        return True, ""
    except Exception as e:
        return False, f"OPF failed: {e}"


def check_security_criteria(net):
    vm = net.res_bus.vm_pu
    volt_ok = bool((vm >= VMIN_PU - 1e-6).all() and (vm <= VMAX_PU + 1e-6).all())
    if len(net.res_line):
        loading = net.res_line.loading_percent.fillna(0.0)
        line_ok = bool((loading <= LINE_LOADING_LIMIT + 1e-6).all())
    else:
        line_ok = True
    return volt_ok and line_ok


class OutageCtx:
    def __init__(self, net, table, idx): self.net, self.table, self.idx = net, table, idx
    def __enter__(self): getattr(self.net, self.table).at[self.idx, "in_service"] = False
    def __exit__(self, exc_type, exc, tb):
        getattr(self.net, self.table).at[self.idx, "in_service"] = True
        return False


def add_parallel_line_like(net, lid):
    """Add a parallel line with same electrical parameters or std_type."""
    if lid not in net.line.index:
        return None
    ln = net.line.loc[lid]
    # Avoid infinite growth
    if net.line.at[lid, "parallels_added"] >= MAX_PARALLELS_PER_ELEMENT:
        # if cap reached, try uprating instead
        net.line.at[lid, "max_i_ka"] = float(ln.max_i_ka) * LINE_UPRATE_FACTOR
        return "uprated"
    if "std_type" in net.line.columns and isinstance(ln.get("std_type"), str) and ln.std_type:
        new_id = pp.create_line(
            net, from_bus=int(ln.from_bus), to_bus=int(ln.to_bus),
            length_km=float(ln.length_km), std_type=str(ln.std_type),
            name=f"parallel_of_line_{lid}"
        )
    else:
        new_id = pp.create_line_from_parameters(
            net, from_bus=int(ln.from_bus), to_bus=int(ln.to_bus),
            length_km=float(ln.length_km),
            r_ohm_per_km=float(ln.r_ohm_per_km),
            x_ohm_per_km=float(ln.x_ohm_per_km),
            c_nf_per_km=float(ln.c_nf_per_km),
            max_i_ka=float(ln.max_i_ka),
            name=f"parallel_of_line_{lid}"
        )
    net.line.at[lid, "parallels_added"] += 1
    return new_id


def add_parallel_trafo_like(net, tid):
    """Add a parallel transformer with same std_type or parameters."""
    if tid not in net.trafo.index:
        return None
    tr = net.trafo.loc[tid]
    # Avoid infinite growth
    if net.trafo.at[tid, "parallels_added"] >= MAX_PARALLELS_PER_ELEMENT:
        # fallback: increase trafo sn_mva (only if create_from_parameters path used)
        if "sn_mva" in net.trafo.columns:
            net.trafo.at[tid, "sn_mva"] = float(tr.sn_mva) * 1.20
        return "uprated"
    if "std_type" in net.trafo.columns and isinstance(tr.get("std_type"), str) and tr.std_type:
        new_id = pp.create_transformer(
            net, hv_bus=int(tr.hv_bus), lv_bus=int(tr.lv_bus),
            std_type=str(tr.std_type), name=f"parallel_of_trafo_{tid}"
        )
    else:
        # Create from parameters (need core params)
        kwargs = dict(
            hv_bus=int(tr.hv_bus), lv_bus=int(tr.lv_bus),
            sn_mva=float(tr.sn_mva),
            vn_hv_kv=float(tr.vn_hv_kv), vn_lv_kv=float(tr.vn_lv_kv),
            vk_percent=float(getattr(tr, "vk_percent", 10.0)),
            vkr_percent=float(getattr(tr, "vkr_percent", 0.3)),
            pfe_kw=float(getattr(tr, "pfe_kw", 0.0)),
            i0_percent=float(getattr(tr, "i0_percent", 0.0)),
            name=f"parallel_of_trafo_{tid}"
        )
        new_id = pp.create_transformer_from_parameters(net, **kwargs)
    net.trafo.at[tid, "parallels_added"] += 1
    return new_id


def n1_report(net):
    """Run N-1 screening and record violations."""
    rep = {"violations": [], "passed": True}

    def record(kind, idx, detail):
        rep["violations"].append({"type": kind, "index": int(idx), "detail": detail})
        rep["passed"] = False

    line_ids  = list(net.line.index)
    gen_ids   = list(net.gen.index)
    trafo_ids = list(net.trafo.index) if "trafo" in net and len(net.trafo) else []

    # Lines
    for lid in line_ids:
        with OutageCtx(net, "line", lid):
            ok, msg = run_ac_opf(net)
            if not ok:
                record("line", lid, f"OPF infeasible. {msg}")
                continue
            if not check_security_criteria(net):
                overloaded = net.res_line[net.res_line.loading_percent > LINE_LOADING_LIMIT].index.tolist()
                vmin = float(net.res_bus.vm_pu.min()); vmax = float(net.res_bus.vm_pu.max())
                record("line", lid, f"Overloads: {overloaded}; vm_pu [{vmin:.3f},{vmax:.3f}]")

    # Generators
    for gid in gen_ids:
        with OutageCtx(net, "gen", gid):
            ok, msg = run_ac_opf(net)
            if not ok:
                record("gen", gid, f"OPF infeasible. {msg}")
                continue
            if not check_security_criteria(net):
                overloaded = net.res_line[net.res_line.loading_percent > LINE_LOADING_LIMIT].index.tolist()
                vmin = float(net.res_bus.vm_pu.min()); vmax = float(net.res_bus.vm_pu.max())
                record("gen", gid, f"Overloads: {overloaded}; vm_pu [{vmin:.3f},{vmax:.3f}]")

    # Transformers
    for tid in trafo_ids:
        with OutageCtx(net, "trafo", tid):
            ok, msg = run_ac_opf(net)
            if not ok:
                record("trafo", tid, f"OPF infeasible. {msg}")
                continue
            if not check_security_criteria(net):
                overloaded = net.res_line[net.res_line.loading_percent > LINE_LOADING_LIMIT].index.tolist()
                vmin = float(net.res_bus.vm_pu.min()); vmax = float(net.res_bus.vm_pu.max())
                record("trafo", tid, f"Overloads: {overloaded}; vm_pu [{vmin:.3f},{vmax:.3f}]")

    return rep


def parse_overloaded_lines(detail):
    """Extract overloaded line IDs from violation detail text."""
    try:
        part = detail.split("Overloads:", 1)[1]
        arr  = part.split("[", 1)[1].split("]", 1)[0]
        ids = []
        for t in arr.split(","):
            t = t.strip()
            if t:
                ids.append(int(t))
        return ids
    except Exception:
        return []


def reinforce_once(net, violations):
    """
    Root-cause aware reinforcement:
      - If OPF infeasible due to *line outage*: add parallel to that line
      - If OPF infeasible due to *trafo outage*: add parallel to that trafo
      - If OPF infeasible due to *gen outage*: relax OTHER gens' max_p
      - If feasible but overloaded lines: add parallel to those overloaded lines (or uprate)
    Returns number of changes applied.
    """
    changes = 0
    lines_to_parallel_from_overload = set()
    need_expand_gens = False

    for v in violations:
        t, idx, detail = v["type"], v["index"], v["detail"]

        if "OPF infeasible" in detail:
            if t == "line":
                res = add_parallel_line_like(net, idx)
                changes += 1 if res is not None else 0
            elif t == "trafo":
                res = add_parallel_trafo_like(net, idx)
                changes += 1 if res is not None else 0
            elif t == "gen":
                need_expand_gens = True

        elif "Overloads:" in detail:
            for lid in parse_overloaded_lines(detail):
                lines_to_parallel_from_overload.add(lid)

    # Handle overloads (feasible cases)
    for lid in lines_to_parallel_from_overload:
        if ADD_PARALLEL_LINES:
            res = add_parallel_line_like(net, lid)
            changes += 1 if res is not None else 0
        else:
            if lid in net.line.index:
                net.line.at[lid, "max_i_ka"] = float(net.line.at[lid, "max_i_ka"]) * LINE_UPRATE_FACTOR
                changes += 1

    # Handle gen expansion (exclude outaged one by nature of context)
    if need_expand_gens and len(net.gen):
        net.gen["max_p_mw"] *= GEN_MAXP_UPFACTOR
        changes += 1

    return changes


def reinforce_until_n1_secure(net, max_iter=MAX_ITER):
    for it in range(1, max_iter + 1):
        # Ensure base case is feasible (slightly relax gens if needed)
        ok, _ = run_ac_opf(net)
        if not ok or not check_security_criteria(net):
            net.gen["max_p_mw"] *= GEN_MAXP_UPFACTOR

        rep = n1_report(net)
        if rep["passed"]:
            return True, rep, net

        changed = reinforce_once(net, rep["violations"])
        if changed == 0:
            # No actionable changes; stop early
            return False, rep, net

    # Final check after reaching iteration cap
    rep = n1_report(net)
    return rep["passed"], rep, net


def pretty_summary(rep, limit=20):
    if rep["passed"]:
        return "✅ N-1 PASSED."
    out = ["⚠️  N-1 NOT PASSED. Sample violations:"]
    for v in rep["violations"][:limit]:
        out.append(f"  - outage type={v['type']}, idx={v['index']}: {v['detail']}")
    return "\n".join(out)


# -----------------------
# Main
# -----------------------
if __name__ == "__main__":
    # 1) Base RTS-24
    base = build_base_rts24()

    # 2) Add renewables from your real data (optional)
    if RENEWABLE_PLANTS:
        add_renewables(base, RENEWABLE_PLANTS)

    # 3) Deep copy & reinforce
    net = copy.deepcopy(base)
    ok, report, reinforced = reinforce_until_n1_secure(net, max_iter=MAX_ITER)
    print(pretty_summary(report))

    # 4) Save
    out_file = "rts24_n1_reinforced_with_res_rootcause.json"
    pp.to_json(reinforced, out_file)
    print(f"Saved: {out_file}")


✅ N-1 PASSED.
Saved: rts24_n1_reinforced_with_res_rootcause.json
